<a href="https://colab.research.google.com/github/paymantohidifar/gpt2-text-classifier-from-scratch/blob/dev/notebooks/interactive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **GPT-2 Text Classifier**

---

## **Google Colab Setup**

In [1]:
import os
import subprocess
import sys

# 1. Environment Detection: Check sys.modules for Colab runtime
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("Detected Google Colab environment. Initializing setup...")

    # Define paths
    repo_url = (
        "https://github.com/paymantohidifar/gpt2-text-classifier-from-scratch.git"
    )
    target_dir = "/content/gpt2-classifier"

    # Clean stale builds and clone main branch
    subprocess.run(f"rm -rf {target_dir}", shell=True, check=True)
    subprocess.run(
        f"git clone {repo_url} --branch main {target_dir}",
        shell=True,
        check=True,
    )

    # Change working directory
    os.chdir(target_dir)

    # Bootstrap uv and install dependencies into system environment
    install_cmd = (
        'curl -LsSf https://astral.sh/uv/install.sh | sh && '
        'export PATH="$HOME/.local/bin:${PATH}" && '
        'uv pip install -e .[gpu,dev] --system --break-system-packages --color never'
    )
    subprocess.run(install_cmd, shell=True, check=True)
    print("Colab setup complete.")

else:
    # Local Development: Enable IPython Auto-Reload safely
    ipython = get_ipython()  # Built-in IPython execution context
    if ipython is not None:
        ipython.run_line_magic("load_ext", "autoreload")
        ipython.run_line_magic("autoreload", "2")
        print("Enabled IPython autoreload.")

Detected Google Colab environment. Initializing setup...
Colab setup complete.


In [3]:
!rm -rf /content/gpt2-classifier
!git clone https://github.com/paymantohidifar/gpt2-text-classifier-from-scratch.git --branch dev gpt2-classifier
%cd gpt2-classifier

# Bootstrap uv globally and pull GPU-enabled binaries directly into the system layer
!curl -LsSf https://astral.sh/uv/install.sh | sh && \
export PATH="$HOME/.local/bin:${PATH}" && \
uv pip install -e .[gpu,dev] \
        --system \
        --break-system-packages \
        --color never

Cloning into 'gpt2-classifier'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 188 (delta 87), reused 171 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (188/188), 278.39 KiB | 2.55 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/gpt2-classifier
downloading uv 0.11.32 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.12.13 environment at: /usr
Resolved 140 packages in 786ms
Prepared 29 packages in 1m 09s
Uninstalled 20 packages in 697ms
Installed 29 packages in 424ms
 + async-lru==2.3.0
 + comm==0.2.3
 ~ gpt2-classifier==0.1.0 (from file:///content/gpt2-classifier)
 - ipykernel==6.17.1
 + ipykernel==7.3.0
 + jedi==0.20.0
 + json5==0.15.0
 + jupyter-builder==1.1.1
 - jupyter-client==7.4.9
 + jupyter-client==8.9.1
 + jupyter-lsp==2.3.1
 + jupyterlab==4.6.2
 + jupyterlab-server==2.28.0
 + nest-asyncio2==1.7.

Verify installed and configured PyTorch:

In [29]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.6.0+cu124
True


---

## **SMS-Spam Collection**

In [30]:
from gpt2_classifier.datasets_registry import get_dataset_spec
from gpt2_classifier.data import prepare_dataset

In [ ]:
spec = get_dataset_spec("sms-spam")
print(spec)

In [ ]:
path = prepare_dataset(spec)
print("File path:", path)

In [1]:
from gpt2_classifier.data import create_data_loaders

train_loader, val_loader, test_loader = create_data_loaders("sms-spam")

In [2]:
for input_batch, target_batch in train_loader:
    pass

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

Input batch dimensions: torch.Size([8, 120])
Label batch dimensions torch.Size([8])


In [3]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

130 training batches
19 validation batches
38 test batches


In [3]:
from gpt2_classifier import paths
from gpt2_classifier.config import get_model_config, URL_DIR
from gpt2_classifier.weights import download_and_load_gpt2, load_weights_into_gpt
from gpt2_classifier.model import GPTModel


model_name = "gpt2-small (124M)"
model_config = get_model_config(model_name)
model_url = f"https://huggingface.co/openai-community/{URL_DIR[model_name]}/resolve/main/model.safetensors"
model_destination = paths.MODELS_DIR / f"{URL_DIR[model_name]}.safetensors"

state_dict = download_and_load_gpt2(model_url, model_destination)
model = GPTModel(model_config)
load_weights_into_gpt(model, state_dict)

The model already exists and is up-to-date: /content/gpt2-classifier/models/gpt2.safetensors


In [4]:
print(model)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [6]:
from gpt2_classifier.utils import generate_response

text_1 = "Every effort moves you"

reponse = generate_response(
    text=text_1,
    model=model,
    max_new_tokens=15
)

print(reponse)

Every effort moves you forward.

The first step is to understand the importance of your work


In [5]:
for param in model.parameters():
    param.requires_grad = False

import torch

torch.manual_seed(123)

num_classes = 2

model.out_head = torch.nn.Linear(in_features=model_config['emb_dim'], out_features=num_classes)


for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

In [6]:
print(model)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [41]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print(inputs)

tensor([[5211,  345,  423,  640]])


In [42]:
with torch.no_grad():
    outputs = model(inputs)
print(outputs)
print(outputs.shape)

tensor([[[-1.5854,  0.9904],
         [-3.7235,  7.4548],
         [-2.2661,  6.6049],
         [-3.5983,  3.9902]]])
torch.Size([1, 4, 2])


In [43]:
from gpt2_classifier.utils import get_device
from gpt2_classifier.evaluate import calc_accuracy_loader

device = get_device()
model.to(device)

torch.manual_seed(123)

train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 46.25%
Validation accuracy: 45.00%
Test accuracy: 48.75%


In [26]:
from gpt2_classifier.evaluate import calc_loss_loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

Training loss: 2.453
Validation loss: 2.583
Test loss: 2.322


In [16]:
# import time
# from gpt2_classifier.train import train_classifier_simple

# start_time = time.time()

# torch.manual_seed(123)

# optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

# num_epochs = 5
# train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
#     model, train_loader, val_loader, optimizer, device,
#     num_epochs=num_epochs, eval_freq=50, eval_iter=5,
# )

# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 2.153, Val loss 2.392
Ep 1 (Step 000050): Train loss 0.617, Val loss 0.637
Ep 1 (Step 000100): Train loss 0.523, Val loss 0.557
Training accuracy: 70.00% | Validation accuracy: 72.50%
Ep 2 (Step 000150): Train loss 0.561, Val loss 0.489
Ep 2 (Step 000200): Train loss 0.419, Val loss 0.397
Ep 2 (Step 000250): Train loss 0.409, Val loss 0.353
Training accuracy: 82.50% | Validation accuracy: 85.00%
Ep 3 (Step 000300): Train loss 0.333, Val loss 0.320
Ep 3 (Step 000350): Train loss 0.340, Val loss 0.306
Training accuracy: 90.00% | Validation accuracy: 90.00%
Ep 4 (Step 000400): Train loss 0.136, Val loss 0.200
Ep 4 (Step 000450): Train loss 0.153, Val loss 0.132
Ep 4 (Step 000500): Train loss 0.222, Val loss 0.137
Training accuracy: 100.00% | Validation accuracy: 97.50%
Ep 5 (Step 000550): Train loss 0.207, Val loss 0.143
Ep 5 (Step 000600): Train loss 0.083, Val loss 0.074
Training accuracy: 100.00% | Validation accuracy: 97.50%
Training completed in 1.00 mi

In [7]:
import torch
import torch.nn as nn

def get_adam_param_groups(model: nn.Module, weight_decay: float = 0.1):
    """
    Filters trainable parameters and splits them into weight-decay
    and no-weight-decay groups (excluding 1D tensors like biases/LayerNorm).
    """
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        # 1. Skip non-trainable parameters completely
        if not param.requires_grad:
            continue

        # 2. Separate 1D parameters (biases, norms) from 2D+ weight matrices
        if param.ndim >= 2:
            decay_params.append(param)
        else:
            no_decay_params.append(param)

    optim_groups = [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]
    return optim_groups

In [8]:
import time
from gpt2_classifier.utils import get_device
from gpt2_classifier.train import train_classifier_simple

device = get_device()
model.to(device)

start_time = time.time()

torch.manual_seed(123)

# Usage inside training setup:
optim_groups = get_adam_param_groups(model, weight_decay=0.1)
optimizer = torch.optim.AdamW(optim_groups, lr=5e-5)

# optimizer = torch.optim.AdamW(
#     (p for p in model.parameters() if p.requires_grad),
#     lr=5e-5,
#     weight_decay=0.1
# )

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 2.153, Val loss 2.392
Ep 1 (Step 000050): Train loss 0.617, Val loss 0.637
Ep 1 (Step 000100): Train loss 0.523, Val loss 0.557
Training accuracy: 70.00% | Validation accuracy: 72.50%
Ep 2 (Step 000150): Train loss 0.561, Val loss 0.489
Ep 2 (Step 000200): Train loss 0.419, Val loss 0.397
Ep 2 (Step 000250): Train loss 0.409, Val loss 0.353
Training accuracy: 82.50% | Validation accuracy: 85.00%
Ep 3 (Step 000300): Train loss 0.334, Val loss 0.320
Ep 3 (Step 000350): Train loss 0.340, Val loss 0.307
Training accuracy: 90.00% | Validation accuracy: 90.00%
Ep 4 (Step 000400): Train loss 0.138, Val loss 0.202
Ep 4 (Step 000450): Train loss 0.154, Val loss 0.133
Ep 4 (Step 000500): Train loss 0.223, Val loss 0.136
Training accuracy: 100.00% | Validation accuracy: 97.50%
Ep 5 (Step 000550): Train loss 0.206, Val loss 0.143
Ep 5 (Step 000600): Train loss 0.083, Val loss 0.073
Training accuracy: 100.00% | Validation accuracy: 97.50%
Training completed in 1.04 mi

The discrepency between two notebooks are likely datset distributions. Copy data from book's directory here and rerun training.